# Nagpur FloodGuard: Explainable Flood Susceptibility Engine

Creates `clean_rainfall.csv`, `clean_elevation.tif`, `risk_features.csv`, `risk_model.py`, `risk_scores.geojson`, and `model_explanation.md`.

This is a flood **susceptibility/risk** model, not a flood forecast. Use validated Government of India OGD/RMC rainfall, SRTM DEM, and IFI-Impacts or verified local flood records before publishing outputs.

## Inputs

Upload: (1) rainfall CSV with `date`, `rainfall_mm`, optionally `latitude`, `longitude`; (2) Nagpur-clipped SRTM GeoTIFF; (3) flood-events CSV with `latitude`, `longitude`.

Suggested sources: OGD District-wise Rainfall of Nagpur Division, RMC Nagpur Rainfall Activity, NASA/USGS SRTM DEM, and India Flood Inventory / IFI-Impacts.

In [1]:
!pip -q install geopandas rasterio pyogrio
from pathlib import Path
import json, shutil
import numpy as np
import pandas as pd
import geopandas as gpd
from shapely.geometry import box
import rasterio
from rasterio.mask import mask

OUT = Path('floodguard_output'); OUT.mkdir(exist_ok=True)
NAGPUR_BOUNDS = (79.00, 21.05, 79.20, 21.23)  # west, south, east, north
GRID_SIZE_M = 1_000
print('Output:', OUT.resolve())

Output: /content/floodguard_output


In [2]:
# Paste direct public download URLs here, or upload the provided ZIP / individual files.
from pathlib import Path
from urllib.request import urlretrieve
import zipfile

RAINFALL_URL = ''       # Direct CSV URL from OGD/RMC, not a catalogue page
DEM_URL = ''            # Direct Nagpur-clipped SRTM GeoTIFF URL
FLOOD_EVENTS_URL = ''   # Direct IFI-Impacts/local-events CSV URL

def fetch(url, filename):
    if not url:
        return None
    path, _ = urlretrieve(url, filename)
    return path

RAINFALL_CSV = fetch(RAINFALL_URL, 'nagpur_rainfall.csv')
DEM_TIF = fetch(DEM_URL, 'nagpur_srtm.tif')
FLOOD_EVENTS_CSV = fetch(FLOOD_EVENTS_URL, 'nagpur_flood_events.csv')

# Upload a ZIP or individual files when direct URLs are unavailable.
if not any((RAINFALL_CSV, DEM_TIF, FLOOD_EVENTS_CSV)):
    from google.colab import files
    uploaded = files.upload()
    for name in uploaded:
        if name.lower().endswith('.zip'):
            with zipfile.ZipFile(name) as archive:
                archive.extractall('floodguard_inputs')
    candidates = list(Path('.').rglob('*.csv'))
    tif_candidates = list(Path('.').rglob('*.tif')) + list(Path('.').rglob('*.tiff'))
    RAINFALL_CSV = next((str(path) for path in candidates if any(key in path.name.lower() for key in ('rain', 'chirps', 'precip'))), None)
    DEM_TIF = str(tif_candidates[0]) if tif_candidates else None
    FLOOD_EVENTS_CSV = next((str(path) for path in candidates if 'flood' in path.name.lower()), None)

# Resolve the bundled evidence package after an optional upload or ZIP extraction.
ZONE_FILE = next((candidate for candidate in (Path('NEW_floodguard_real_evidence_package/risk_zones.geojson'), Path('floodguard_inputs/risk_zones.geojson'), Path('/content/NEW_floodguard_real_evidence_package/risk_zones.geojson')) if candidate.is_file()), None)
if ZONE_FILE is None:
    raise FileNotFoundError('Missing NEW_floodguard_real_evidence_package/risk_zones.geojson. Upload or mount the evidence package, then rerun this cell.')
risk_zones = gpd.read_file(ZONE_FILE)
print('Risk zones:', len(risk_zones), '|', ZONE_FILE.resolve())
print({'RAINFALL_CSV': RAINFALL_CSV, 'DEM_TIF': DEM_TIF, 'FLOOD_EVENTS_CSV': FLOOD_EVENTS_CSV})

KeyboardInterrupt: 

In [ ]:
# Synthetic data is opt-in only. Real rainfall and DEM inputs are required for deliverables.
rng = np.random.default_rng(42)
def demo_inputs():
    rain = pd.DataFrame({'date': pd.date_range('2024-06-01', periods=300, freq='D'),
                         'rainfall_mm': rng.gamma(2, 18, 300),
                         'longitude': rng.uniform(79.00, 79.20, 300),
                         'latitude': rng.uniform(21.05, 21.23, 300)})
    events = pd.DataFrame({'longitude': rng.uniform(79.03, 79.15, 25),
                           'latitude': rng.uniform(21.08, 21.19, 25)})
    return rain, events
USE_SYNTHETIC_DEMO = False
rainfall, _ = demo_inputs() if USE_SYNTHETIC_DEMO else (None, None)
if RAINFALL_CSV: rainfall = pd.read_csv(RAINFALL_CSV)
if rainfall is None: raise ValueError('Set RAINFALL_CSV to a real rainfall CSV, or explicitly set USE_SYNTHETIC_DEMO=True for testing.')
events = pd.read_csv(FLOOD_EVENTS_CSV) if FLOOD_EVENTS_CSV else None
required = {'date', 'rainfall_mm'}
if not required.issubset(rainfall.columns): raise ValueError(f'Rainfall requires {required}')
rainfall['date'] = pd.to_datetime(rainfall.date, errors='coerce')
rainfall['rainfall_mm'] = pd.to_numeric(rainfall.rainfall_mm, errors='coerce')
rainfall = rainfall.dropna(subset=['date','rainfall_mm']).query('rainfall_mm >= 0').copy()
rainfall.to_csv(OUT/'clean_rainfall.csv', index=False)
assert not rainfall.empty, 'No valid rainfall rows'
print('Clean rainfall rows:', len(rainfall))

In [ ]:
# Make 1 km zones in Nagpur's local UTM CRS.
study = gpd.GeoDataFrame(geometry=[box(*NAGPUR_BOUNDS)], crs=4326).to_crs(32644)
xmin, ymin, xmax, ymax = study.total_bounds
cells = [box(x,y,min(x+GRID_SIZE_M,xmax),min(y+GRID_SIZE_M,ymax))
         for x in np.arange(xmin,xmax,GRID_SIZE_M) for y in np.arange(ymin,ymax,GRID_SIZE_M)]
zones = gpd.GeoDataFrame({'zone_id':[f'Z{i:03d}' for i in range(1,len(cells)+1)]}, geometry=cells, crs=32644)
zones = zones[zones.intersects(study.geometry.iloc[0])].copy()
zones['geometry'] = zones.geometry.intersection(study.geometry.iloc[0])
zones = zones[~zones.geometry.is_empty & (zones.area > 0)].copy()
# Keep annual rainfall for provenance, but score short-duration flood triggers.
def rainfall_event_features(daily):
    daily = daily.sort_values(['zone_id', 'date']).copy()
    daily['rolling_3day_mm'] = daily.groupby('zone_id').rainfall_mm.transform(lambda values: values.rolling(3, min_periods=1).sum())
    return daily.groupby('zone_id').agg(
        rainfall_mm=('rainfall_mm', 'sum'),
        max_daily_rainfall_mm=('rainfall_mm', 'max'),
        max_3day_rainfall_mm=('rolling_3day_mm', 'max'),
    )
zones['rainfall_mm'] = rainfall.rainfall_mm.mean()
daily_total = rainfall.groupby('date', as_index=False).rainfall_mm.mean()
daily_total['zone_id'] = 'study_area'
fallback = rainfall_event_features(daily_total).iloc[0]
zones['max_daily_rainfall_mm'] = fallback.max_daily_rainfall_mm
zones['max_3day_rainfall_mm'] = fallback.max_3day_rainfall_mm
if {'latitude','longitude'}.issubset(rainfall):
    # Assign every zone its nearest rainfall pixel; a 'within' join left ~96% of zones dry.
    daily = rainfall.sort_values(['latitude','longitude','date']).copy()
    daily['rolling_3day_mm'] = daily.groupby(['latitude','longitude']).rainfall_mm.transform(lambda v: v.rolling(3, min_periods=1).sum())
    pixels = daily.groupby(['latitude','longitude'], as_index=False).agg(
        rainfall_mm=('rainfall_mm','sum'),
        max_daily_rainfall_mm=('rainfall_mm','max'),
        max_3day_rainfall_mm=('rolling_3day_mm','max'))
    pix_gdf = gpd.GeoDataFrame(pixels, geometry=gpd.points_from_xy(pixels.longitude, pixels.latitude), crs=4326).to_crs(32644)
    joined = gpd.sjoin_nearest(zones[['zone_id','geometry']], pix_gdf, how='left')
    for col in ('rainfall_mm','max_daily_rainfall_mm','max_3day_rainfall_mm'):
        zones[col] = zones.zone_id.map(joined.set_index('zone_id')[col]).fillna(0)
print('Zones:',len(zones))

In [ ]:
# Derive elevation and slope from an SRTM DEM. If no DEM is supplied, outputs are flagged synthetic.
zones['elevation_m'] = np.nan; zones['slope_deg'] = np.nan
if DEM_TIF:
    with rasterio.open(DEM_TIF) as src:
        if src.crs is None: raise ValueError('DEM needs a CRS')
        dem_crs = zones.to_crs(src.crs)
        sample_points = zones.geometry.centroid.to_crs(src.crs)
        data = src.read(1, masked=True)
        valid = data.compressed()
        if not valid.size:
            raise ValueError('DEM has no valid elevation pixels.')
        data = data.filled(np.median(valid)).astype(float)
        # Calculate one DEM-wide slope raster, then sample each zone centroid.
        y_res = abs(src.transform.e) * 111_320
        mean_lat = (NAGPUR_BOUNDS[1] + NAGPUR_BOUNDS[3]) / 2
        x_res = abs(src.transform.a) * 111_320 * np.cos(np.deg2rad(mean_lat))
        gy, gx = np.gradient(data, y_res, x_res)
        slope = np.degrees(np.arctan(np.hypot(gx, gy)))
        for idx, point in sample_points.items():
            row, col = src.index(point.x, point.y)
            if 0 <= row < src.height and 0 <= col < src.width:
                zones.loc[idx, 'elevation_m'] = data[row, col]
                zones.loc[idx, 'slope_deg'] = slope[row, col]
    shutil.copy(DEM_TIF, OUT/'clean_elevation.tif')
elif USE_SYNTHETIC_DEMO:
    zones['elevation_m'] = rng.uniform(260, 340, len(zones))
    zones['slope_deg'] = rng.uniform(0, 8, len(zones))
    # ponytail: a small raster keeps the demo's output contract testable; replace with SRTM for real work.
    from rasterio.transform import from_bounds
    transform = from_bounds(*NAGPUR_BOUNDS, 32, 32)
    with rasterio.open(OUT/'clean_elevation.tif', 'w', driver='GTiff', height=32, width=32, count=1, dtype='float32', crs='EPSG:4326', transform=transform) as dst:
        dst.write(np.full((32,32), 300, dtype='float32'), 1)
    print('WARNING: synthetic elevation/slope demo only.')
else:
    raise ValueError('Set DEM_TIF to a real Nagpur DEM, or explicitly set USE_SYNTHETIC_DEMO=True for testing.')
if zones.elevation_m.isna().any():
    raise ValueError('DEM does not cover every Nagpur analysis zone. Use a DEM covering the full study area.')
zones['slope_deg'] = zones.slope_deg.fillna(0.0)

In [ ]:
# Only verified, supplied event records may influence historical risk.
zones['historical_event_count'] = 0
zones['historical_data_available'] = events is not None
if events is not None:
    if not {'latitude','longitude'}.issubset(events): raise ValueError('Flood events require latitude and longitude')
    event_gdf = gpd.GeoDataFrame(events, geometry=gpd.points_from_xy(events.longitude,events.latitude), crs=4326).to_crs(32644)
    counts = gpd.sjoin(event_gdf,zones[['zone_id','geometry']],predicate='within').groupby('zone_id').size()
    zones['historical_event_count'] = zones.zone_id.map(counts).fillna(0).astype(int)
    print('Verified events mapped to zones:', zones.historical_event_count.sum())
else:
    hotspot_geo = next((p for p in (Path('NEW_floodguard_real_evidence_package/flood_hotspots.geojson'), Path('/content/NEW_floodguard_real_evidence_package/flood_hotspots.geojson')) if p.is_file()), None)
    if hotspot_geo is None:
        print('No verified flood events supplied: historical risk is excluded from the score.')
    else:
        hot = gpd.read_file(hotspot_geo).to_crs(32644)
        counts = gpd.sjoin(hot, zones[['zone_id','geometry']], predicate='within').groupby('zone_id').size()
        zones['historical_event_count'] = zones.zone_id.map(counts).fillna(0).astype(int)
        zones['historical_data_available'] = True
        print('Hotspot events mapped to zones:', zones.historical_event_count.sum())

In [ ]:
# Explainable model. Match this scoring contract in risk_model.py.
def pct(s, ascending=True): return s.rank(pct=True, ascending=ascending).fillna(0.5)
weights = {'max_daily_rainfall': .25, 'max_3day_rainfall': .25, 'low_elevation': .30, 'low_slope': .20}
zones['max_daily_rainfall_factor'] = pct(zones.max_daily_rainfall_mm)
zones['max_3day_rainfall_factor'] = pct(zones.max_3day_rainfall_mm)
zones['low_elevation_factor'] = pct(zones.elevation_m, ascending=False)
zones['low_slope_factor'] = pct(zones.slope_deg, ascending=False)
zones['low_lying'] = zones.elevation_m <= zones.elevation_m.quantile(.25)
zones['historical_factor'] = 0.0
if zones.historical_data_available.astype(str).str.lower().eq('true').all():
    zones['historical_factor'] = pct(zones.historical_event_count)
    weights['historical'] = .20
weight_total = sum(weights.values())
for factor, weight in weights.items():
    zones[f'{factor}_contribution'] = (100 * weight / weight_total) * zones[f'{factor}_factor']
zones['risk_score'] = zones[[f'{factor}_contribution' for factor in weights]].sum(axis=1).round().astype(int)
zones['risk_class'] = pd.cut(zones.risk_score, [-1,30,55,75,100], labels=['LOW','MODERATE','HIGH','CRITICAL'])
features = zones.drop(columns='geometry').copy()
features.to_csv(OUT/'risk_features.csv', index=False)
zones.to_crs(4326).to_file(OUT/'risk_scores.geojson', driver='GeoJSON')
features[['zone_id','rainfall_mm','max_daily_rainfall_mm','max_3day_rainfall_mm','elevation_m','slope_deg','low_lying','historical_event_count','risk_score','risk_class']].head()

In [ ]:
# Save the model as a portable Python module. Keep this source identical to the checked-in scorer.
model = '''"""Explainable, relative flood-susceptibility scoring for Nagpur zones."""

import pandas as pd


REQUIRED_COLUMNS = ("rainfall_mm", "elevation_m", "slope_deg")
RISK_CLASSES = ["LOW", "MODERATE", "HIGH", "CRITICAL"]


def _percentile(series, ascending=True):
    return series.rank(pct=True, ascending=ascending).fillna(0.5)


def score(features):
    """Return a scored copy of a zone feature table with audit contributions."""
    missing = set(REQUIRED_COLUMNS) - set(features.columns)
    if missing:
        raise ValueError(f"Missing required columns: {', '.join(sorted(missing))}")

    out = features.copy()
    for column in REQUIRED_COLUMNS:
        out[column] = pd.to_numeric(out[column], errors="coerce")
    if out[list(REQUIRED_COLUMNS)].isna().any().any():
        raise ValueError("Rainfall, elevation, and slope must be numeric and present.")
    if (out[list(REQUIRED_COLUMNS)] < 0).any().any():
        raise ValueError("Rainfall, elevation, and slope cannot be negative.")

    weights = {
        "rainfall": 0.40,
        "low_elevation": 0.25,
        "low_slope": 0.15,
    }
    out["rainfall_factor"] = _percentile(out["rainfall_mm"])
    out["low_elevation_factor"] = _percentile(out["elevation_m"], ascending=False)
    out["low_slope_factor"] = _percentile(out["slope_deg"], ascending=False)
    out["low_lying"] = out["elevation_m"] <= out["elevation_m"].quantile(0.25)

    historical_available = out.get("historical_data_available", pd.Series(False, index=out.index))
    historical_available = historical_available.astype(str).str.lower().eq("true")
    use_historical = historical_available.all()
    out["historical_data_available"] = historical_available
    out["historical_factor"] = 0.0
    if use_historical:
        if "historical_event_count" not in out:
            raise ValueError("historical_event_count is required when historical data is available.")
        out["historical_event_count"] = pd.to_numeric(out["historical_event_count"], errors="coerce")
        if out["historical_event_count"].isna().any() or (out["historical_event_count"] < 0).any():
            raise ValueError("historical_event_count must be present and non-negative.")
        out["historical_factor"] = _percentile(out["historical_event_count"])
        weights["historical"] = 0.20

    total_weight = sum(weights.values())
    for factor, weight in weights.items():
        out[f"{factor}_contribution"] = (100 * weight / total_weight) * out[f"{factor}_factor"]
    contribution_columns = [f"{factor}_contribution" for factor in weights]
    out["risk_score"] = out[contribution_columns].sum(axis=1).round().astype(int)
    out["risk_class"] = pd.cut(out["risk_score"], [-1, 30, 55, 75, 100], labels=RISK_CLASSES)
    return out
'''
model = model.replace('REQUIRED_COLUMNS = ("rainfall_mm", "elevation_m", "slope_deg")', 'REQUIRED_COLUMNS = ("max_daily_rainfall_mm", "max_3day_rainfall_mm", "elevation_m", "slope_deg")').replace('Rainfall, elevation, and slope', 'Rainfall-event features, elevation, and slope').replace('"rainfall": 0.40,\n        "low_elevation": 0.25,\n        "low_slope": 0.15,', '"max_daily_rainfall": 0.25,\n        "max_3day_rainfall": 0.25,\n        "low_elevation": 0.30,\n        "low_slope": 0.20,').replace('out["rainfall_factor"] = _percentile(out["rainfall_mm"])', 'out["max_daily_rainfall_factor"] = _percentile(out["max_daily_rainfall_mm"])\n    out["max_3day_rainfall_factor"] = _percentile(out["max_3day_rainfall_mm"])')
(OUT/'risk_model.py').write_text(model)
explanation = '# FloodGuard model explanation\n\n## Purpose\nAn explainable flood susceptibility score, not a flood prediction model.\n\n## Formula\nWith verified flood events: `score = 100 × (0.40 rainfall + 0.25 low-elevation + 0.15 low-slope + 0.20 historical-events)`. Without verified events, the available three weights are rescaled to total 1.0.\n\n## Inputs and provenance\n`clean_rainfall.csv` comes from the supplied rainfall CSV. The bundled input is CHIRPS v2 daily precipitation for the Nagpur area, 2024-01-01 to 2024-12-31, extracted from https://data.chc.ucsb.edu/products/CHIRPS-2.0/.\n\n`clean_elevation.tif` comes from the supplied DEM. The bundled Nagpur-area EPSG:4326 DEM was built from Mapzen Terrain Tiles: https://registry.opendata.aws/terrain-tiles/.\n\n## Outputs\n`historical_data_available` records whether historical evidence affected the score. `low_lying` marks the lowest elevation quartile.\n\n## Classes\n0–30 LOW, 31–55 MODERATE, 56–75 HIGH, 76–100 CRITICAL.\n\n## Limits\nScores are relative to this study area and input period. Validate against documented flood events before operational use.\n'
explanation = explanation.replace('Without verified events, the available three weights are rescaled to total 1.0.\n\n## Inputs', 'Without verified events, the available three weights are rescaled to total 1.0.\n\nEach factor is normalized as a percentile within the current study-area run. Lower elevation and lower slope receive higher factor values. The score is therefore comparative, not an absolute probability of flooding.\n\n## Inputs').replace('`historical_data_available` records whether historical evidence affected the score. `low_lying` marks the lowest elevation quartile.', '`historical_data_available` records whether historical evidence affected the score. `low_lying` marks the lowest elevation quartile. The `*_factor` columns show normalized inputs and the `*_contribution` columns show each factor\'s contribution in score points; their sum, rounded to an integer, is `risk_score`.').replace('Scores are relative to this study area and input period. Validate against documented flood events before operational use.\n', 'Scores are relative to this study area and input period. Validate against documented flood events before operational use.\n\nRainfall is currently aggregated over the supplied 2024 CHIRPS period. Before operational use, add event-window features such as 1-day maximum rainfall and 3-day accumulation, then validate those features against verified local flood events.\n')
explanation = explanation.replace('With verified flood events: `score = 100 × (0.40 rainfall + 0.25 low-elevation + 0.15 low-slope + 0.20 historical-events)`. Without verified events, the available three weights are rescaled to total 1.0.', 'Without verified flood events: `score = 100 × (0.25 maximum-1-day-rainfall + 0.25 maximum-3-day-rainfall + 0.30 low-elevation + 0.20 low-slope)`. With verified events, the available weights are rescaled to include a 0.20 historical-event factor.').replace('`historical_data_available` records whether historical evidence affected the score. `low_lying` marks the lowest elevation quartile.', '`max_daily_rainfall_mm` and `max_3day_rainfall_mm` capture short-duration rainfall extremes. `rainfall_mm` remains the annual total for provenance. `historical_data_available` records whether historical evidence affected the score. `low_lying` marks the lowest elevation quartile. The `*_factor` columns show normalized inputs and the `*_contribution` columns show each factor contribution in score points.').replace('Scores are relative to this study area and input period. Validate against documented flood events before operational use.', 'Scores are relative to this study area and input period. The event features are derived from the supplied 2024 CHIRPS period. Validate their thresholds and score weights against verified local flood events before operational use.')
(OUT/'model_explanation.md').write_text(explanation)
print(*[p.name for p in OUT.iterdir()], sep='\n')


In [ ]:
# Download every deliverable as one archive.
archive = shutil.make_archive('nagpur_floodguard_deliverables', 'zip', OUT)
files.download(archive)